# AIML-02 — Spleen 3D U-Net (Colab T4)

MONAI 3D U-Net on **MSD Task 09 (Spleen)**. Trains from CT volumes with a foreground-focused patch sampler and evaluates with sliding-window Dice.

- Data: ~1.5 GB tar from the Medical Segmentation Decathlon.
- Runtime: 60 epochs with resume; a disconnect costs at most one epoch.
- Output: `unet_model.pt` + `unet_metrics.json`.

> Education and research only. Not a medical device.


## 1. GPU check

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU and rerun.'


## 2. Install MONAI and fetch the data

In [ ]:
!pip install -q "monai>=1.4" nibabel
!wget -q --show-progress https://msd-for-monai.s3-us-west-2.amazonaws.com/Task09_Spleen.tar
!tar -xf Task09_Spleen.tar -C /content
!find /content/Task09_Spleen -name '._*' -delete
!ls /content/Task09_Spleen | head
!ls /content/Task09_Spleen/imagesTr | head
!ls /content/Task09_Spleen/labelsTr | head


## 3. Clone the repo

In [ ]:
!rm -rf /content/nexvia
!git clone --depth 1 https://github.com/OmkarAla/nexvia.git /content/nexvia

## 4. Mount Google Drive

Checkpoints go to Drive so a disconnect costs at most one epoch. This run uses a larger patch (96×96×64), manual per-volume Dice, and a foreground-fraction diagnostic.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/nexvia/unet-artifacts"
print('artifacts persist at /content/drive/MyDrive/nexvia/unet-artifacts')

## 4. Train

In [ ]:
!cd /content/nexvia && python AIML-02/training/train_unet.py \n  --data /content/Task09_Spleen \n  --out "/content/drive/MyDrive/nexvia/unet-artifacts-v2" \n  --epochs 100 \n  --resume \n  --run-id colab-unet-003


## 5. Inspect and download

In [ ]:
!cd "/content/drive/MyDrive/nexvia" && zip -r unet-artifacts-v2.zip unet-artifacts-v2
from google.colab import files
files.download("/content/drive/MyDrive/nexvia/unet-artifacts-v2.zip")


## Notes

- The default patch size is `(64, 64, 32)` with channels `(16, 32, 64, 128)`.
- Evaluation uses `sliding_window_inference` with 50% overlap and discrete Dice.
- Artifacts go to **Google Drive**; `--resume` continues from `unet_last.pt`.
- After a disconnect: rerun the GPU check, the data cell if `/content` was cleared, the clone cell, the Drive mount, then the training cell. It resumes automatically.
- Place `unet_model.pt` in `AIML-02/artifacts/demo/` to enable CT segmentation in the worker.
